In [35]:
import pandas as pd
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()

df = pd.DataFrame([
    ("Welfare", 246.1), ("Health", 241.8), ("State pensions", 137.8),
    ("Debt interest", 124.7), ("Education", 118.7), ("Defence", 63.6),
    ("Public order & safety", 51.4), ("Transport", 46.4),
    ("Business & industry", 40.4), ("Government admin", 23.0),
    ("Housing & utilities", 22.3), ("Environment", 17.1),
    ("Culture", 14.5), ("Overseas aid", 7.7),
], columns=["fn", "spend"])
df["hi"] = df["fn"] == "Debt interest"
order = df.sort_values("spend", ascending=False)["fn"].tolist()

# shared x so bars and labels use the same scale; extra headroom so "246" isn't clipped
x = alt.X("spend:Q", title=None, axis=None, scale=alt.Scale(domain=[0, 290]))

base = alt.Chart(df).encode(
    y=alt.Y("fn:N", sort=order, title=None,
            scale=alt.Scale(paddingInner=0.25, paddingOuter=0.15),
            # smaller category labels (delete labelFontSize to use the theme's own size)
            axis=alt.Axis(labelFontSize=10, labelLimit=220)))

bars = base.mark_bar(cornerRadius=2).encode(
    x=x,
    # theme palette: teal for the ordinary bars, accent red for the highlighted one
    color=alt.Color("hi:N", scale=alt.Scale(domain=[False, True]), legend=None))

labels = base.mark_text(align="left", dx=5, fontSize=10, color="#676A86").encode(
    x=x, text=alt.Text("spend:Q", format=".0f"))

caption = alt.Title(
    text="Source: HMRC Annual Tax Summary / PESA Table 5.2, 2024–25",
    subtitle=[
        "UK public spending by function, £ billion.",
        "Debt interest is the fourth-largest line — above education, nearly double defence.",
    ],
    orient="bottom", anchor="start",
    fontSize=11, subtitleFontSize=10,
    color="#676A86", subtitleColor="#676A86", dy=12)

chart = (
    (bars + labels)
    .properties(
        width=400, height=300,
        # fit everything (labels, caption, padding) inside 400 x 360
        autosize=alt.AutoSizeParams(type="fit", contains="padding"),
        # room on the right so the longest value label can't be clipped
        padding={"left": 5, "right": 20, "top": 5, "bottom": 5})
    .configure(background="white", font="Circular Std")
    .configure_view(fill="transparent", stroke="transparent"))

styles.save(chart, path=".", name="debt_interest_spending", svg=True)
chart

alt.LayerChart(...)

In [28]:
import numpy as np
import pandas as pd
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()

SHAPE = "coin"          # "coin" (one circle = £1) or "square"

df = pd.DataFrame([
    ("Welfare", 246.1), ("Health", 241.8), ("State pensions", 137.8),
    ("Debt interest", 124.7), ("Education", 118.7), ("Defence", 63.6),
    ("Public order & safety", 51.4), ("Transport", 46.4),
    ("Business & industry", 40.4), ("Government admin", 23.0),
    ("Housing & utilities", 22.3), ("Environment", 17.1),
    ("Culture", 14.5), ("Overseas aid", 7.7),
], columns=["fn", "spend"])

# Denominator = the 14 functions shown. If you want shares of an official total
# (e.g. total managed expenditure), replace TOTAL with that figure.
TOTAL = df["spend"].sum()

# Only the story gets a colour: debt interest vs the two comparators, everything else is grey.
SPEND = df.set_index("fn")["spend"]
groups = pd.Series({
    "Debt interest": SPEND["Debt interest"],
    "Education": SPEND["Education"],
    "Defence": SPEND["Defence"],
})
groups["Everything else"] = TOTAL - groups.sum()

# Round to exactly 100 coins (largest-remainder method)
raw = groups / TOTAL * 100
n = np.floor(raw).astype(int)
for name in (raw - n).sort_values(ascending=False).index[: 100 - n.sum()]:
    n[name] += 1

# tone + opacity per group (red = theme accent, teal = theme primary, slate = theme text grey)
STYLE = {
    "Debt interest":   ("red",   1.00),
    "Education":       ("teal",  1.00),
    "Defence":         ("slate", 0.75),
    "Everything else": ("slate", 0.20),
}

# --- 10 x 10 grid, filled left-to-right, top-to-bottom, story first ---
cells = [g for g in groups.index for _ in range(n[g])]
grid_df = pd.DataFrame({"group": cells,
                        "cx": [i % 10 + 0.5 for i in range(100)],
                        "cy": [i // 10 + 0.5 for i in range(100)]})
grid_df["tone"] = grid_df["group"].map(lambda g: STYLE[g][0])
grid_df["alpha"] = grid_df["group"].map(lambda g: STYLE[g][1])

# --- key (right-hand side, lower half) ---
key = pd.DataFrame({"group": list(groups.index)})
key["cx"] = 11.0
key["yc"] = [5.6 + 1.35 * k for k in range(len(key))]
key["amount"] = ["£" + str(n[g]) for g in key["group"]]
key["tone"] = key["group"].map(lambda g: STYLE[g][0])
key["alpha"] = key["group"].map(lambda g: STYLE[g][1])

X = alt.Scale(domain=[0, 20], nice=False)
Y = alt.Scale(domain=[0, 10], reverse=True, nice=False)
# teal / red come from the theme palette (in that order); the grey is your caption grey
theme_colour = alt.Color("tone:N", scale=alt.Scale(domain=["teal", "red"]), legend=None)
colour = alt.condition(alt.datum.tone == "slate", alt.value("#676A86"), theme_colour)
opacity = alt.Opacity("alpha:Q", scale=None, legend=None)

shape = "circle" if SHAPE == "coin" else "square"
size = 230 if SHAPE == "coin" else 290

def dots(data, cx="cx", cy="cy", sz=size):
    return alt.Chart(data).mark_point(shape=shape, filled=True, size=sz, strokeWidth=0).encode(
        x=alt.X(f"{cx}:Q", scale=X, axis=None), y=alt.Y(f"{cy}:Q", scale=Y, axis=None),
        color=colour, opacity=opacity)

grid = dots(grid_df)
key_marks = dots(key, cy="yc", sz=size * 0.8)

def text(data, xpos, ypos, value, size_, bold=False, colour_enc=None, align="left", baseline="middle"):
    enc = colour_enc if colour_enc is not None else alt.value("#676A86")
    return (alt.Chart(data).mark_text(align=align, baseline=baseline, fontSize=size_,
                                      fontWeight="bold" if bold else "normal")
            .encode(x=alt.X(datum=xpos, scale=X), y=alt.Y(datum=ypos, scale=Y),
                    text=value, color=enc))

# --- big callout: the "for every £100" line ---
call = pd.DataFrame({"tone": ["red"]})
big = (alt.Chart(call).mark_text(align="left", baseline="alphabetic", fontSize=44, fontWeight="bold")
       .encode(x=alt.X(datum=11.0, scale=X), y=alt.Y(datum=2.1, scale=Y),
               text=alt.value(f"£{n['Debt interest']}"), color=theme_colour))
sub1 = text(call, 11.0, 3.1, alt.value("of every £100 spent"), 12)
sub2 = text(call, 11.0, 3.85, alt.value("goes on debt interest"), 12, bold=True)

# --- key text ---
key_amt = (alt.Chart(key).mark_text(align="left", baseline="middle", fontSize=12, fontWeight="bold")
           .encode(x=alt.X(datum=12.0, scale=X), y=alt.Y("yc:Q", scale=Y), text="amount:N",
                   color=alt.value("#676A86")))
key_name = (alt.Chart(key).mark_text(align="left", baseline="middle", fontSize=12)
            .encode(x=alt.X(datum=13.6, scale=X), y=alt.Y("yc:Q", scale=Y), text="group:N",
                    color=alt.value("#676A86")))

caption = alt.Title(
    text="Source: HMRC Annual Tax Summary / PESA Table 5.2, 2024–25",
    subtitle=[
        f"UK public spending by function. Each {'coin' if SHAPE == 'coin' else 'square'} is £1 in every £100.",
        "Debt interest is more than education and nearly double defence.",
        f"Shares are of the £{TOTAL:,.1f}bn spent across the 14 functions shown.",
    ],
    orient="bottom", anchor="start",
    fontSize=11, subtitleFontSize=10,
    color="#676A86", subtitleColor="#676A86", dy=12)

chart = (
    alt.layer(grid, key_marks, big, sub1, sub2, key_amt, key_name)
    .properties(width=400, height=300,
                padding={"left": 5, "right": 5, "top": 5, "bottom": 5})
    .configure(background="white", font="Circular Std")
    .configure_view(fill="transparent", stroke="transparent"))

styles.save(chart, path=".", name="debt_interest_coins", svg=True)
chart

alt.LayerChart(...)